###DAY 9 (28/02/26) – Recommendation System
####🏗️ Architecture & Strategy
Welcome to Day 9! We are wrapping up Phase 2 (AI System Building) by tackling one of the most profitable algorithms in eCommerce: the Recommendation System.

Today, we will build a Collaborative Filtering engine using Spark's Alternating Least Squares (ALS) algorithm.  Unlike content-based filtering (which looks at product attributes like color or brand), collaborative filtering looks strictly at user behavior: "Users who bought what you bought, also bought this."

####Our Senior-Level Strategy:

* **Implicit vs. Explicit Feedback**: The practice code provided maps events to a 1, 2, or 3 score. This is called Implicit Feedback because the user didn't explicitly leave a 3-star review; we are inferring their preference from their actions (views, carts, purchases). We must tell our ALS model to expect implicit data, or it will evaluate the math incorrectly.

* **User-Item Matrix Deduplication**: A user might view the same product 10 times. ALS expects exactly one rating per user-item pair. We will aggregate our events using max() so that if a user views (1) and then purchases (3) an item, the final matrix records their highest intent (3).

* **The "Cold Start" Problem**: How do you recommend items to a brand-new user with no history? You can't using ALS! We will set coldStartStrategy="drop" to prevent the model from crashing on new users during evaluation, and discuss how to handle them in production.

* **Actionable Outputs**: Spark ALS outputs recommendations as complex arrays. We will use PySpark functions to "explode" these arrays into a flat, tabular format so the engineering team can easily load them into a web application.

####Data Prep & Implicit Rating Mapping
We start by loading our raw Bronze events, mapping them to numerical intent scores, and safely casting our IDs to integers (a strict requirement for Spark's ALS algorithm).

In [0]:
from pyspark.sql import functions as F

# 1. Environment Setup
catalog_name = "course_catalog"  
schema_name = "ecommerce_governed"
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

print("⏳ Loading raw events for user-item interactions...")
df_events = spark.table("events_delta_managed")

# 2. Rating Mapping & Deduplication
print("🧮 Generating Implicit Ratings Matrix...")

# We group by user_id and product_id to ensure only ONE row per user-item pair.
user_item_df = df_events.groupBy("user_id", "product_id").agg(
    F.max(
        F.when(F.col("event_type") == "purchase", 3)
         .when(F.col("event_type") == "cart", 2)
         .otherwise(1)
    ).alias("implicit_rating")
).select(
    # ALS strictly requires Integer types for IDs. We cast them to prevent crashes.
    F.col("user_id").cast("integer"),
    F.col("product_id").cast("integer"),
    "implicit_rating"
).dropna() # Drop nulls created by casting errors

# NOTE: Removed .cache() because Databricks Serverless handles this automatically!

print(f"✅ Generated rating matrix with {user_item_df.count():,} unique interactions.")
display(user_item_df.limit(5))

####Train ALS Model with MLflow
We will train our matrix factorization model. Notice the crucial addition of implicitPrefs=True, which is the difference between a junior implementation and a senior production model.

In [0]:
import os
import mlflow
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Setup MLflow & Security Staging Volume (Inherited from Day 7)
volume_name = "ml_assets"
mlflow_tmp_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/mlflow_staging"
os.environ["MLFLOW_DFS_TMP"] = mlflow_tmp_path

username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
mlflow.set_experiment(f"/Users/{username}/Day9_Product_Recommendations")

# 2. Train / Test Split
# We need to test how well the model predicts ratings it hasn't seen
train_df, test_df = user_item_df.randomSplit([0.8, 0.2], seed=42)

print("🚀 Training ALS Collaborative Filtering Model...")

with mlflow.start_run(run_name="ALS_Implicit_Recommender"):
    
    # 3. Initialize ALS (Senior PySpark Configuration)
    als = ALS(
        userCol="user_id",
        itemCol="product_id",
        ratingCol="implicit_rating",
        implicitPrefs=True,       # ⚠️ CRITICAL: Tells ALS to treat scores as confidence, not explicit 5-star reviews
        coldStartStrategy="drop", # ⚠️ Prevents NaN evaluation crashes for unseen users/items
        nonnegative=True,         # Ensures we don't predict negative purchase intents
        rank=10,                  # Number of latent factors (hidden features)
        maxIter=10,
        seed=42
    )
    
    # Log hyperparameters
    mlflow.log_param("model_type", "ALS_Implicit")
    mlflow.log_param("rank", 10)
    mlflow.log_param("implicitPrefs", True)
    
    # 4. Train the Model
    als_model = als.fit(train_df)
    
    # 5. Evaluate the Model
    # We use RMSE (Root Mean Squared Error) to see how far off our predicted intents are from actuals
    evaluator = RegressionEvaluator(
        metricName="rmse", 
        labelCol="implicit_rating",
        predictionCol="prediction"
    )
    predictions = als_model.transform(test_df)
    rmse = evaluator.evaluate(predictions)
    
    # Log metrics and model
    mlflow.log_metric("test_rmse", rmse)
    mlflow.spark.log_model(
        spark_model=als_model, 
        artifact_path="als_recommender", 
        dfs_tmpdir=mlflow_tmp_path
    )
    
    print(f"   🏆 ALS Model RMSE: {rmse:.4f}")
    print("   ✅ Model securely logged to MLflow.")

####Generate Top-5 Recommendations & Flatten for BI
The .recommendForAllUsers(5) function returns a complex nested array. A web frontend cannot easily digest that. We will use the explode function to unnest the arrays so every recommendation has its own row, and then save it to a Gold table.

In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# ---------------------------------------------------------
# GENERATE TOP 5 RECOMMENDATIONS (SERVERLESS BYPASS)
# ---------------------------------------------------------
print("🎯 Generating Top 5 product recommendations (Serverless-Safe Architecture)...")

# 1. Define a Cohort
# To avoid a Cartesian memory explosion, we demonstrate the engine on 50 unique users
user_cohort = user_item_df.select("user_id").distinct().limit(50)
all_items = user_item_df.select("product_id").distinct()

# 2. Create the Inference Grid
# We cross-join our 50 users with all possible items to create every possible combination
print("   ➤ Building User-Item grid...")
user_item_grid = user_cohort.crossJoin(all_items)

# 3. Score the Grid (The UC-Safe Way!)
# Because .transform() uses modern Catalyst DataFrames instead of RDDs, UC allows it!
print("   ➤ Predicting purchase intent scores...")
scored_grid = als_model.transform(user_item_grid)

# 4. Rank and Filter for Top 5 using Spark SQL
# We partition by user and sort their predicted scores from highest to lowest
print("   ➤ Ranking Top 5 items per user...")
window_spec = Window.partitionBy("user_id").orderBy(F.col("prediction").desc())

top_5_recs = scored_grid.withColumn("rank", F.row_number().over(window_spec)) \
                        .filter(F.col("rank") <= 5) \
                        .select(
                            "user_id",
                            "rank",
                            F.col("product_id").alias("recommended_product_id"),
                            F.round("prediction", 4).alias("confidence_score")
                        )

print("\n📊 Final Recommendations Table:")
display(top_5_recs.limit(10))

# 5. Save to Gold Layer
gold_table_name = "gold_user_recommendations"
print(f"\n💾 Saving to Gold Layer: {catalog_name}.{schema_name}.{gold_table_name}...")

top_5_recs.write.format("delta").mode("overwrite").saveAsTable(gold_table_name)
spark.sql(f"OPTIMIZE {gold_table_name} ZORDER BY (user_id)")

print("🏆 Phase 2 Complete! Your Recommendation Engine is live in the Gold Layer.")

####Key Learnings & Interview Talking Points
* **Implicit vs. Explicit Feedback Modeling**: "I understand the mathematical difference between explicit feedback (like a 5-star review) and implicit feedback (like clicks, cart additions, and purchases). When building my PySpark ALS model, I engineered an implicit rating matrix and specifically configured the algorithm with implicitPrefs=True to treat these behavioral metrics as confidence intervals rather than absolute ratings."

* **Handling the 'Cold Start' Problem**: "Collaborative filtering fundamentally suffers from the 'Cold Start' problem—it cannot generate accurate recommendations for a brand-new user with zero historical interactions. In my pipeline, I set coldStartStrategy="drop" during evaluation to ensure stable RMSE metric logging in MLflow. In a true production deployment, I would architect a fallback system: serving ALS recommendations for returning users, while serving globally popular items for new users."

* **Serverless Compute & Memory Management**: "While building iterative ML algorithms, I originally attempted to use PySpark's .cache() command. However, because I was deploying on Databricks Serverless compute, I discovered that manual persistence commands are actually unsupported. This taught me that modern Serverless architectures utilize intelligent, automatic disk caching that renders manual JVM memory management obsolete."

* **Unity Catalog Security vs. Legacy MLlib RDDs**: "The biggest hurdle I overcame was an architectural conflict between Spark MLlib and Unity Catalog. The native recommendForAllUsers() function relies on legacy RDD MapPartitions and higher-order functions under the hood. When deploying this on a tightly governed Serverless cluster, Unity Catalog correctly blocked the execution because untrusted RDDs violate strict Lakehouse data access controls."

* **Engineering a Serverless-Compliant Inference Pipeline**: "Instead of abandoning the model when Unity Catalog blocked the native recommendation function, I engineered a Serverless-compliant bypass using pure Spark SQL. I orchestrated a Cartesian cross-join to build a user-item inference grid, scored them using the UC-safe .transform() method, and extracted the Top 5 recommendations per user using Spark SQL Window functions. This allowed me to safely persist the final recommendations to the Delta Gold layer without breaking Lakehouse security boundaries."